# The GL rating engine, one file at a time

**Nineteen notebooks, one per Python file in `gl_engine/`. Every example runs.**

The engine is about 5,000 lines and it rates General Liability in 51 jurisdictions. It manages that without a single line of per-state code, because it doesn't implement ISO's rules — it **executes** them. ISO publishes its rating content as a program, and this is an interpreter for that program.

That one fact explains most of what looks strange in here. There is no deductible module, no territory logic, no Georgia branch. Search the whole engine for `deductible` and you get one hit, and it is a referral message.

Start at [`01-config`](01-config.ipynb) and read down. Each notebook assumes the ones above it.

## Check your setup first

The engine reads ISO's licensed content from outside this repository. Without it, most notebooks will not run.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

from gl_engine import config

root = config.CORPUS_ROOT
print("python      :", sys.version.split()[0])
print("corpus root :", root)
print("corpus found:", root.exists())

if root.exists():
    n = len([d for d in root.iterdir()
             if d.is_dir() and d.name not in config.EXCLUDE_DIRS])
    print(f"             {n} jurisdiction directories")
else:
    print()
    print("  Set GL_ERC_ROOT to your copy of ISO's General Liability corpus.")
    print("  Notebooks 02, 03, 08 and 09 will still run without it.")

In [ ]:
# One end-to-end check: if this prints a premium, everything works.
from gl_engine.rating import Kernel

r = Kernel().rate("../Engine_Payloads/GA/submission.json")
print("Georgia premium:", r.premium, "from", r.packages)

## The reading order

Dependency order, not alphabetical.

### Foundations

| | | |
|---|---|---|
| [01](01-config.ipynb) | `config.py` | Where ISO's content lives, and the date the engine refuses to go below |
| [02](02-errors.ipynb) | `errors.py` | Every way it is allowed to fail. **Read this early — the refusals are the design** |
| [03](03-domain-cell.ipynb) | `domain/cell.py` | A value that cannot name its ISO source cannot be constructed |

### Finding and reading ISO's content

| | | |
|---|---|---|
| [04](04-erc-discovery.ipynb) | `erc/discovery.py` | Identity comes from the namespace, never the folder name |
| [05](05-resolve-resolver.ipynb) | `resolve/resolver.py` | Which two packages govern a state on a date. Five steps, five wrong shortcuts |
| [06](06-resolve-book.ipynb) | `resolve/book.py` | State overrides countrywide by name — even when the override is empty |
| [07](07-erc-tables.ipynb) | `erc/tables.py` | Definition first, then data. Shape and population are separate axes |

### The interpreter

| | | |
|---|---|---|
| [08](08-interp-values.ipynb) | `interp/values.py` | Five types, and a null that is not zero |
| [09](09-interp-tree.ipynb) | `interp/tree.py` | ISO's rules have no arguments. They read and write one shared document |
| [10](10-interp-program.ipynb) | `interp/program.py` | Rules indexed by name, and the entry point every census missed |
| [11](11-interp-nodes.ipynb) | `interp/nodes.py` | **ISO's whole vocabulary — 54 instructions.** One of them does the insurance |
| [12](12-interp-interpreter.ipynb) | `interp/interpreter.py` | Frames, dispatch, lookup, rounding, trace |

### Submissions and rating

| | | |
|---|---|---|
| [13](13-schema-fields.ipynb) | `schema/fields.py` | The submission format, read from ISO rather than designed |
| [14](14-schema-validate.ipynb) | `schema/validate.py` | Findings, not exceptions — and each check says what it does *not* cover |
| [15](15-rating-submission.ipynb) | `rating/submission.py` | JSON onto the tree ISO's rules expect |
| [16](16-rating-kernel.ipynb) | `rating/kernel.py` | **Submission in, rating out.** The front door |
| [17](17-rating-referrals.ipynb) | `rating/referrals.py` | When the engine must not answer |

### Proving and driving it

| | | |
|---|---|---|
| [18](18-assertions.ipynb) | `assertions.py` | Load-time checks that fail. None of them warns |
| [19](19-cli.ipynb) | `cli.py` | Five verbs, and none of them rate |

## If you only have twenty minutes

[**05**](05-resolve-resolver.ipynb) — which rules apply, and why "the latest one" is wrong.
[**11**](11-interp-nodes.ipynb) — the 54 instructions. This is the idea the whole engine rests on.
[**16**](16-rating-kernel.ipynb) — a premium, and the trace that explains it.

## The shape of every notebook

The same six cells each time: **what the file is for** · **its public surface**, generated from the module so it cannot drift · **the smallest thing that works** · **the interesting case**, being whatever that file exists to get right · **what it refuses** · **try it yourself**, with answers at the bottom.

The refusals cell is not filler. This engine stops rather than guessing in a lot of places, and the exception text is usually the clearest statement of a module's contract you will find.

## Two things to know

**Outputs are stripped before commit.** A notebook that has run holds ISO's licensed numbers inside its JSON, and this repository deliberately excludes ISO content. You run them to see the numbers. See [`README.md`](README.md).

**They are tested.** `python tests/verify_notebooks.py` executes every cell of every notebook and fails on any exception — so a notebook that stops matching the code becomes a red test rather than quietly wrong documentation. It has already caught seven mistakes in this set.

## The QA harness has its own set

Everything above is the engine. `qa.py`, `variants.py`, `sweep.py` and the rest of the test
programme live in `scripts/` and `ui/`, and they get a separate set at
[`harness/00-index`](harness/00-index.ipynb) rather than being folded in here — on purpose, so
this set keeps making one claim: no rating concept lives outside the interpreter.